In [1]:
!pip install langchain
!pip install unstructured
!pip install openai
!pip install Cython
!pip install tiktoken

In [ ]:
!pip install --upgrade langchain-astradb

In [3]:
!pip install langchain langchain-openai datasets pypdf

In [4]:
!pip install pdf2image

In [5]:
!pip install pdfminer.six

In [ ]:
!pip install unstructured[pdf]

In [19]:
!sudo apt-get install poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 35 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.10 [186 kB]
Fetched 186 kB in 0s (1,891 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package poppler-utils.
(Reading database ... 1263

In [7]:
!pip install langchain_community

In [20]:
import os
from getpass import getpass

from datasets import (
    load_dataset
)
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.document_loaders import UnstructuredPDFLoader
from langchain.indexes import VectorstoreIndexCreator

In [21]:
import os
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

embedding = OpenAIEmbeddings()

In [22]:
embedding = OpenAIEmbeddings()

# Using Unstructured for loading Multiple Pdfs

In [39]:
root_dir="/content/"

In [40]:
pdf_folder_path = f'{root_dir}/docs/'

In [41]:
os.listdir(pdf_folder_path)

['Scaling Test Time Compute.pdf', 'LangGraph_Slides.pdf', 'Slides.pdf']

In [42]:
# location of the pdf file/files.
loaders = [UnstructuredPDFLoader(os.path.join(pdf_folder_path, fn)) for fn in os.listdir(pdf_folder_path)]

In [43]:
loaders

In [44]:
index = VectorstoreIndexCreator(embedding=embedding).from_loaders(loaders)

In [45]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0)

response = index.query(
    'What is the StateGraph, ToolNode in LangGraph?',
    llm=llm
)
print(response)


In LangGraph, a StateGraph is a class used to build and compile the graph structure. It manages nodes, edges, and the overall state to ensure proper workflow and data flow. It is like a blueprint of a building, defining the structure and flow of the workflow.

A ToolNode in LangGraph is a special kind of node that runs a tool. It connects the tool's output back into the state so that other nodes can use that information. An analogy for ToolNode is an operator using a machine to perform a specific task.


In [46]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0)

answer = index.query_with_sources(
    'What is the StateGraph, ToolNode in LangGraph?',
    llm=llm
)
print(answer)


{'question': 'What is the StateGraph, ToolNode in LangGraph?', 'answer': "The StateGraph in LangGraph is a class used to build and compile the graph structure, managing nodes, edges, and overall state. The ToolNode in LangGraph is a special kind of node that runs a tool and connects the tool's output back into the State. \n", 'sources': '/content//docs/LangGraph_Slides.pdf'}


# Pypdf loader with Multiple Pdfs.

In [47]:
from langchain_astradb import AstraDBVectorStore

In [ ]:
from langchain_astradb import AstraDBVectorStore
ASTRA_DB_API_ENDPOINT="put astra db endpoint here"
ASTRA_DB_APPLICATION_TOKEN="your token here"
ASTRA_DB_KEYSPACE="default_keyspace"

In [49]:
root_dir="/content/"
pdf_folder_path = f'{root_dir}/docs/'
pdfs=os.listdir(pdf_folder_path)

In [50]:
pdfs

['Scaling Test Time Compute.pdf', 'LangGraph_Slides.pdf', 'Slides.pdf']

In [51]:
data=PyPDFLoader("/content/docs/Scaling Test Time Compute.pdf")

In [52]:
data

In [53]:
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)

In [54]:
data.load_and_split(text_splitter=splitter)

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '', 'author': '', 'keywords': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/docs/Scaling Test Time Compute.pdf', 'total_pages': 37, 'page': 0, 'page_label': '1'}, page_content='2024-8-7\nScaling LLM Test-Time Compute Optimally can\nbe More Effective than Scaling Model Parameters\nCharlie Snell♦, 1, Jaehoon Lee2, Kelvin Xu♣, 2 and Aviral Kumar♣, 2\n♣Equal advising,1UC Berkeley,2Google DeepMind,♦Work done during an internship at Google DeepMind\nEnabling LLMs to improve their outputs by using more test-time computation is a critical step towards\nbuilding generally self-improving agents that can operate on open-ended natural language. In this paper,'),
 Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '', '

In [55]:
docs=[]
for pdf in pdfs:
  data=PyPDFLoader(f"/content/docs/{pdf}")
  docs.append(data)

In [56]:
docs_from_pdf = []
for loader in docs:
    docs_from_pdf.extend(loader.load_and_split(text_splitter=splitter))


In [57]:
vstore = AstraDBVectorStore(
    embedding=embedding,
    collection_name="astra_vector_demo",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=ASTRA_DB_KEYSPACE,
)

In [58]:
print(f"Documents from PDF: {len(docs_from_pdf)}.")
inserted_ids_from_pdf = vstore.add_documents(docs_from_pdf)
print(f"Inserted {len(inserted_ids_from_pdf)} documents.")

Documents from PDF: 368.
Inserted 368 documents.


In [59]:
retriever = vstore.as_retriever(search_kwargs={"k": 3})

In [66]:
prompt_template = """
You are a AI philosopher that draws insights from documents related to "Inference time Compute Scaling in LLMs",
"Langgraph Concepts and graphs", "Llama2" to craft well-thought answers to user questions. Use the provided context as the basis
for your answers and do not make up new reasoning paths - just mix-and-match what you are given.
Your answers should be detailed one and strictly related to the provided context.

CONTEXT:
{context}

QUESTION: {question}

YOUR ANSWER:"""

In [67]:
prompt_template = ChatPromptTemplate.from_template(prompt_template)

In [68]:
llm = ChatOpenAI()

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [69]:
chain.invoke("How to Scale Test-Time Computation Optimally and what will be scaling strategy?")

"Scaling test-time computation optimally involves effectively allocating resources to enhance performance under a given compute budget. The strategy involves considering a tradeoff between sequential (revisions) and parallel (best-of-N) test-time computation. The ideal ratio of sequential to parallel testing depends on the compute budget and the difficulty of the specific question being addressed.\n\nFor easier questions, purely sequential test-time compute is beneficial, while harder questions often require a combination of sequential and parallel computation. By refining the proposal distribution with revisions, we can improve the efficacy of the approach, tailoring it to the difficulty of the problem faced by the base LLM's capabilities.\n\nImplementing a compute-optimal scaling strategy involves adapting the allocation of test-time compute based on the prompt at hand. This adaptive, prompt-dependent approach aims to enhance performance while using resources efficiently. By honing i

In [70]:
chain.invoke("What is StateGraph, MessageGraph is LangGraph?")

"In the context of Langgraph concepts and graphs, StateGraph and MessageGraph are components of LangGraph. StateGraph represents the current state of a language model, including its internal representations and memory. It captures the model's knowledge and understanding at a given point in time. On the other hand, MessageGraph focuses on the communication and interaction between different parts of the language model. It deals with how information is passed between components, such as nodes or modules, within the model. MessageGraph plays a crucial role in coordinating the flow of data and decisions within the language model, contributing to its overall functionality and performance. Both StateGraph and MessageGraph are essential elements of LangGraph, contributing to the complex structure and operation of language models in processing natural language tasks."

In [71]:
chain.invoke("What’s the difference between the absolute positional encodings and the relative ones? and also Explain RoPE embeddings in detailed way")

"In the context of language models like LLMs, positional encodings are crucial for helping the model understand the sequential order of tokens in a sentence. Absolute positional encodings assign unique values to each position in the input sequence, regardless of the content of the tokens. On the other hand, relative positional encodings consider the relative distance between tokens, which can be particularly useful for capturing dependencies between tokens that are further apart.\n\nNow, moving on to explaining Relative Positional Embeddings (RoPE) in detail, RoPE is a technique used in transformer-based models to incorporate relative positional information between tokens. Rather than relying solely on absolute positions, RoPE embeddings enable the model to take into account the relative positions of tokens in the sequence, which can be beneficial for tasks requiring understanding of long-range dependencies.\n\nRoPE embeddings are typically calculated based on the relative distance bet

# Directory loaders(Chat With Multiple Doc)

In [ ]:
!rm -rf "/content/docs/.ipynb_checkpoints"

In [ ]:
%pip install langchain_community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.9/302.9 kB 24.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 15.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.5/142.5 kB 18.8 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 24.0
    Uninstalling packaging-24.0:
      Successfully uninstalled packaging-24.0


In [ ]:
!pip install unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.8/433.8 kB 36.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 33.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 53.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 27.1 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993227 sha256=1f09b72aaa02dc2ea6c1dbaa05856ac91c8fb233003bdb4614e30a8bc025eecc
  Stored in directory: /root/.cache/pip/wheels/95/03/7d/59ea870c70ce4e5a370638b5462a7711ab78fba2f655d05106
Successfully built langdetect


In [ ]:
!pip install "unstructured[pdf]"

In [ ]:
!sudo apt-get update

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [110 kB]
Hit:7 https://ppa.launchpadcontent.net/c2d4u.team/c2d4u4.0+/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 110 kB in 1s (86.3 kB/s)
Reading package lists... Done


In [ ]:
!sudo apt-get install poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.4).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.


In [ ]:
!sudo apt-get install libleptonica-dev tesseract-ocr libtesseract-dev python3-pil tesseract-ocr-eng tesseract-ocr-script-latn


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libarchive-dev libimagequant0 libraqm0 mailcap mime-support python3-olefile
  tesseract-ocr-osd
Suggested packages:
  python-pil-doc
The following NEW packages will be installed:
  libarchive-dev libimagequant0 libleptonica-dev libraqm0 libtesseract-dev
  mailcap mime-support python3-olefile python3-pil tesseract-ocr
  tesseract-ocr-eng tesseract-ocr-osd tesseract-ocr-script-latn
0 upgraded, 13 newly installed, 0 to remove and 52 not upgraded.
Need to get 40.0 MB of archives.
After this operation, 123 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libarchive-dev amd64 3.6.0-1ubuntu1 [581 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libimagequant0 amd64 2.17.0-1 [34.6 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libleptonica-dev amd64 1.82.0-3build1 [1,5

In [ ]:
!pip install unstructured-pytesseract
!pip install tesseract-ocr

  Preparing metadata (setup.py) ... done
  Created wheel for tesseract-ocr: filename=tesseract_ocr-0.0.1-cp310-cp310-linux_x86_64.whl size=169756 sha256=34897be8ebc098165522b3bf91977750745c010f07b549bdb9781036df6c27cf
  Stored in directory: /root/.cache/pip/wheels/bb/fd/f3/5c231ecbbb80a1fe33204ff3021d99b54ef6daf6f8099311b8
Successfully built tesseract-ocr


In [ ]:
!pip install "unstructured[pptx]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 17.0 MB/s eta 0:00:00


In [ ]:
!pip install langchain_astradb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.4/124.4 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 56.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.4 MB/s eta 0:00:00
  Created wheel for bson: filename=bson-0.5.10-py3-none-any.whl size=11976 sha256=c94e1e293db9f31890191752c13da886d61da3ae13ae67b22a482ab0fde01661
  Stored in directory: /root/.cache/pip/wheels/36/49/3b/8b33954dfae7a176009c4d721a45af56c8a9c1cdc3ee947945
Successfully built bson


In [ ]:
!pip install langchain langchain-openai datasets pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 46.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.3/320.3 kB 33.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 54.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 22.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 17.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 27.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.20.3
    Uninstalling huggingface-hub-0.20.3:
      Successfully uninstalled huggingface-hub-0.20.3


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from langchain_community.document_loaders import DirectoryLoader


In [ ]:
loader = DirectoryLoader('/content/docs')

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)

In [ ]:
docs = loader.load_and_split(text_splitter=splitter)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


yolox_l0.05.onnx:   0%|          | 0.00/217M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Some weights of the model checkpoint at microsoft/table-transformer-structure-recognition were not used when initializing TableTransformerForObjectDetection: ['model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
len(docs)

233

In [ ]:
import os
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings



In [ ]:
import os
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
embedding = OpenAIEmbeddings()

In [ ]:
from langchain_astradb import AstraDBVectorStore
from langchain.indexes import VectorstoreIndexCreator

In [ ]:
ASTRA_DB_API_ENDPOINT="your api endpoint here"
ASTRA_DB_APPLICATION_TOKEN="your token here"
ASTRA_DB_KEYSPACE="default_keyspace"

In [ ]:
vstore = AstraDBVectorStore(
    embedding=embedding,
    collection_name="multidoc_vector",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=ASTRA_DB_KEYSPACE,
)

In [ ]:
inserted_ids = vstore.add_documents(docs)

In [ ]:
print(f"\nInserted {len(inserted_ids)} documents.")


Inserted 233 documents.


In [ ]:
prompt_template = """
You are an AI philosopher drawing insights from the roadmap of "rag," "llama3," and "genai."
Craft thoughtful answers based on this roadmap, mixing and matching existing paths.
Your responses should be concise and strictly related to the provided context.

ROADMAP CONTEXT:
{context}

QUESTION: {question}

YOUR ANSWER:"""

In [ ]:
prompt_template = ChatPromptTemplate.from_template(prompt_template)

In [ ]:
retriever = vstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
retriever

VectorStoreRetriever(tags=['AstraDBVectorStore', 'OpenAIEmbeddings'], vectorstore=<langchain_astradb.vectorstores.AstraDBVectorStore object at 0x7fefaeb23d90>, search_kwargs={'k': 3})

In [ ]:
llm = ChatOpenAI()

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [ ]:
chain.invoke("can you tell me the roadmap of generative ai?")

'The roadmap of Generative AI includes steps for beginners to advanced learners such as prerequisites, fundamentals, core generative models, developing applications with LLMs, projects, practical experience, and miscellaneous topics. It emphasizes continuous learning, setting specific goals, consistent learning, implementation, experimentation, feedback, and the importance of a background in machine learning and deep learning. Mathematics knowledge is also essential for Generative AI.'

In [ ]:
chain.invoke("what is a llama can you tell me some important point on top of it.")

'Llama (Large Language Model Meta AI) is a family of autoregressive large language models released by Meta AI. The latest version is Llama 3, which was released in April 2024. Llama models have shown superior performance compared to other large language models, such as GPT-3 and PaLM. These models range in parameter sizes from 7B to 70B and have been made accessible for both academic and commercial use.'